# Kapitel 8 — Ähnlichkeit messen · Kommentierte Lösung

*Mathe für KI, Schritt für Schritt*, Belk Siri — Begleit-Notebook.

[In Google Colab öffnen](https://colab.research.google.com/github/belksiri/books/blob/main/maths-pour-l-ia-pas-a-pas/notebooks/de/ch08_corrige.ipynb) · Begleitseite: https://belksiri.github.io/books/maths-pour-l-ia-pas-a-pas/de.html

Jede Antwort zeigt zuerst die Rechnung von Hand, wie in den Lösungen des Buchs, und prüft sie dann in Python.

**Inhalt.** Daten · Die Rechnungen der Lektionen 8.1 bis 8.3 · Aufgaben 8.1 bis 8.6

**Bevor Sie Jupyter öffnen:** Die Aufgaben dieses Kapitels werden zuerst auf Papier gelöst, mit einem Taschenrechner. Die Zellen unten dienen dazu, Ihre Rechnungen zu **überprüfen**. Wie im Buch teilen Mittelwerte und Standardabweichungen durch $n$ und werden aus den sechs Referenzverkäufen A bis F berechnet.

**Variablennamen.** Der Code verwendet die englischen Namen des Buchs: `floor_area` (Fläche, in m²), `distance` (Entfernung zum Zentrum, in km), `price` (Preis, in k€), `home` (Name der Wohnung).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Farben des Buchs: Referenzwohnungen in Blau, ausgewählte Nachbarn orange umkreist, Entfernungen in Magenta
BLUE, ORANGE, MAGENTA = '#2a78d6', '#eb6834', '#c2185b'

## Daten

Die Kennungen D2.1 und D2.E2 sind die des Buchs und der CSV-Dateien. Die Spalten werden beim Einlesen ins Englische umbenannt; die Zuordnung steht im Kommentar.

In [ ]:
# Daten von Kapitel 8, aus den CSV-Dateien des Buchs übernommen (nicht ändern)

# D2.1 — die sieben Wohnungen; Spalten der CSV-Datei ins Englische umbenannt:
#   home = logement, floor_area = surface_m2 (m²), distance = distance_km (km), price = prix_kEUR (k€)
homes = pd.DataFrame({
    'home':       ['A', 'B', 'C', 'D', 'E', 'F', 'G'],
    'floor_area': [32, 45, 54, 70, 85, 98, 62],
    'distance':   [3.0, 1.5, 2.5, 1.0, 2.5, 0.5, 2.5],
    'price':      [118, 172, 185, 245, 279, 335, 209],
})
reference = homes[homes['home'] != 'G'].reset_index(drop=True)   # die sechs Referenzverkäufe, A bis F
X = reference[['floor_area', 'distance']].to_numpy(dtype=float)  # eine Zeile pro Verkauf: (m²; km)
price = reference['price'].to_numpy(dtype=float)                 # Preise der Referenzverkäufe, in k€
G = np.array([62, 2.5])                                          # die Wohnung G: 62 m², 2,5 km (verkauft für 209 k€)

# D2.E2 — die Wohnung I (Aufgabe 8.2)
I = np.array([75, 3.5])   # 75 m², 3,5 km

# Beispiele des Buchs (Lektionen 8.1 und 8.3): Stadtplan in km, Nachrichten, erfundene Einbettungen
school, G_map = np.array([4.5, 6]), np.array([1.5, 2])            # die Schule und G auf dem Plan, in km
R1, R2, P = np.array([3, 1]), np.array([6, 2]), np.array([1, 3])  # („Sitzung“, „Angebot“)
embeddings = {'Sitzung': np.array([4, 3]), 'Versammlung': np.array([3, 4]),
              'Rabatt': np.array([-3, 4]), 'Aktion': np.array([-4, 3])}

### Die Variablen der Daten

Die Namen im Code sind englisch. Für die Spalten gibt die letzte Spalte den Namen an, der in den CSV-Dateien des Buchs verwendet wird.

| Name | Bedeutung | Einheit | Spalte der CSV-Datei |
|:--|:--|:--|:--|
| `home` | Kennung der Wohnung (A, B…) | — | `logement` |
| `floor_area` | Wohnfläche der Wohnung | m² | `surface_m2` |
| `price` | Verkaufspreis | k€ | `prix_kEUR` |
| `distance` | Entfernung zum Stadtzentrum | km | `distance_km` |
| `distance_m` | Entfernung zum Stadtzentrum | m | — |
| `message` | Kennung des Textes (T1 bis T4) | — | — |
| `meeting_count` | Anzahl der Vorkommen des Wortes „Sitzung“ | Vorkommen | — |
| `offer_count` | Anzahl der Vorkommen des Wortes „Angebot“ | Vorkommen | — |
| `volunteer_count` | Anzahl der Vorkommen des Wortes „Ehrenamt“ | Vorkommen | — |

In [ ]:
# Bereitgestellte Werkzeuge: Sie müssen sie nicht ändern

def nearest(dists, k=3):
    """Die k nächstgelegenen Referenzverkäufe (Entfernungen in der Reihenfolge A bis F) und ihr mittlerer Preis."""
    order = np.argsort(dists, kind='stable')[:k]
    return list(reference['home'].iloc[order]), float(price[order].mean())


def show_table(dists, label):
    """Tabelle der Abstände zu einer Wohnung, vom kürzesten zum längsten."""
    t = pd.DataFrame({'home': reference['home'], label: np.round(dists, 3), 'price': price})
    print(t.sort_values(label, kind='stable').to_string(index=False))

## Die Rechnungen der Lektionen, in Python

Diese Zellen wiederholen die Rechnungen der Lektionen 8.1 bis 8.3. Führen Sie sie aus und vergleichen Sie mit dem Buch.

In [ ]:
# Lektion 8.1 — Norm und Abstand (Gleichung 8.1)
trip = school - G_map                       # Verschiebung von G zur Schule, in km
print(f"Verschiebung {trip}, Abstand {np.sqrt((trip ** 2).sum()):g} km")
print(f"vom Zentrum zu G: {np.linalg.norm(G_map):g} km")             # die Entfernung zum Zentrum aus Tabelle 2.1
print(f"Norm von (2; 3; 6): {np.linalg.norm([2, 3, 6]):g}")      # zweimal Pythagoras oder eine einzige Rechnung
C_home = X[2]                                                       # C: 54 m², 2,5 km
print(f"Abstand von C zu G: {np.linalg.norm(C_home - G):g} (gemischte Einheiten: m² und km)")

In [ ]:
# Lektion 8.2 — erster Versuch: Rohabstände zu G, dann Entfernung zum Zentrum in Metern
raw_G = np.sqrt(((X - G) ** 2).sum(axis=1))
show_table(raw_G, 'raw_distance')
n_km, e_km = nearest(raw_G)
print(f"3 nächste Nachbarn (roh, km): {n_km}, Schätzung {e_km:.1f} k€")
X_m, G_m = X * [1, 1000], G * [1, 1000]                             # Entfernung zum Zentrum in Metern
n_m, e_m = nearest(np.linalg.norm(X_m - G_m, axis=1))
print(f"3 nächste Nachbarn (roh, m):  {n_m}, Schätzung {e_m:.0f} k€")

In [ ]:
# Lektion 8.2 — zweiter Versuch: mit den Referenzverkäufen standardisieren (Gleichung 8.2)
mu, sigma = X.mean(axis=0), X.std(axis=0)                           # Division durch n (ddof=0)
print(f"Fläche: Mittelwert {mu[0]:.0f} m², Standardabweichung {sigma[0]:.2f} m²")
print(f"Entfernung: Mittelwert {mu[1]:.3f} km, Standardabweichung {sigma[1]:.3f} km (√29/6 = {np.sqrt(29) / 6:.3f})")
Z, z_G = (X - mu) / sigma, (G - mu) / sigma
print('z-Werte von G:', np.round(z_G, 2))
std_G = np.linalg.norm(Z - z_G, axis=1)
show_table(std_G, 'std_distance')
neighbours, estimate = nearest(std_G)
print(f"3 nächste Nachbarn (standardisiert): {neighbours}, Schätzung {estimate:.0f} k€ (tatsächlicher Preis: 209 k€)")

fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(Z[:, 0], Z[:, 1], color=BLUE, s=40)
ax.scatter(*z_G, marker='D', color='black', s=40)
for h, (a, b) in zip(reference['home'], Z):
    ax.annotate(h, (a, b), xytext=(5, 4), textcoords='offset points')
ax.annotate('G', z_G, xytext=(5, 4), textcoords='offset points')
idx = np.argsort(std_G, kind='stable')[:3]
ax.scatter(Z[idx, 0], Z[idx, 1], s=220, facecolors='none', edgecolors=ORANGE, linewidths=1.8)
ax.add_patch(plt.Circle(z_G, std_G[idx[-1]], fill=False, linestyle='--', color='grey'))
ax.set_aspect('equal'); ax.set_xlabel('z Fläche'); ax.set_ylabel('z Entfernung'); ax.grid(alpha=0.4)
plt.show()

In [ ]:
# Lektion 8.3 — Abstand oder Kosinus? (Gleichung 8.3)
def cos_lesson(u, v):
    return float(np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v)))

print(f"Abstände: R1–R2 {np.linalg.norm(R1 - R2):.2f}, R1–P {np.linalg.norm(R1 - P):.2f}")
print(f"Kosinus:  R1–R2 {cos_lesson(R1, R2):.2f}, R1–P {cos_lesson(R1, P):.2f}, "
      f"Winkel R1–P {np.degrees(np.arccos(cos_lesson(R1, P))):.0f}°")

# Ausblick: die Einbettungen aus Tabelle 8.3 (alle mit Norm 5)
for a, b in [('Sitzung', 'Versammlung'), ('Rabatt', 'Aktion'), ('Sitzung', 'Rabatt'), ('Sitzung', 'Aktion')]:
    print(f"cos({a}, {b}) = {cos_lesson(embeddings[a], embeddings[b]):.2f}")
message = (embeddings['Sitzung'] + embeddings['Versammlung']) / 2       # die Nachricht „Sitzung, Versammlung“
print(f"Nachricht {message}: cos mit Sitzung {cos_lesson(message, embeddings['Sitzung']):.2f}, "
      f"mit promotion {cos_lesson(message, embeddings['Aktion']):.2f}")

## Aufgabe 8.1 — Abstände auf dem Stadtplan

*Begriffe: Norm und Abstand (Lektion 8.1, Gleichung 8.1).* Auf dem Plan von Abbildung 8.1, in Kilometern, liegt das Schwimmbad im Punkt $(7{,}5 ; 10)$ und der Bahnhof im Punkt $(-3{,}5 ; 5)$; G liegt im Punkt $(1{,}5 ; 2)$.

### a) bis e) Die Rechnung von Hand

a) $\sqrt{36 + 64} = \sqrt{100} = 10$.

b) Von G zum Schwimmbad beträgt die Verschiebung $(7{,}5 - 1{,}5 ; 10 - 2) = (6 ; 8)$: Der Abstand beträgt nach a) 10 km.

c) Von G zum Bahnhof: $(-5 ; 3)$, und $\sqrt{25 + 9} = \sqrt{34} \approx 5{,}83$ km. Vom Bahnhof zum Schwimmbad: $(11 ; 5)$, und $\sqrt{121 + 25} = \sqrt{146} \approx 12{,}08$ km.

d) Über den Bahnhof: $5{,}83 + 12{,}08 \approx 17{,}91$ km, also etwa 7,91 km mehr als der direkte Weg. Ein Umweg verkürzt einen Weg nie.

e) Am Boden $\sqrt{4 + 9} = \sqrt{13}$, dann $\sqrt{13 + 36} = \sqrt{49} = 7$. Direkt: $\sqrt{4 + 9 + 36} = 7$.

In [ ]:
# Lösung: die Norm eines Vektors, ohne np.linalg.norm (Gleichung 8.1)
def norm(u):
    u = np.asarray(u, dtype=float)
    return float(np.sqrt((u ** 2).sum()))

pool, station = np.array([7.5, 10]), np.array([-3.5, 5])   # Schwimmbad und Bahnhof, in km

In [ ]:
# Prüfung von a) bis e)
if norm([6, 8]) is None:
    print('noch nicht ausgefüllt. Ergänzen Sie die Funktion norm.')
else:
    d_pool, d_station, d_station_pool = norm(pool - G_map), norm(station - G_map), norm(pool - station)
    print(f"a) {norm([6, 8]):g}   b) {d_pool:g} km   c) {d_station:.2f} km und {d_station_pool:.2f} km")
    print(f"d) Umweg: {d_station + d_station_pool - d_pool:.2f} km   e) {norm([2, 3, 6]):g}")
    print('→', 'richtig' if abs(d_pool - 10) < 1e-9 and abs(d_station - 5.831) < 5e-4
          and abs(d_station_pool - 12.083) < 5e-4 and abs(norm([2, 3, 6]) - 7) < 1e-9 else 'bitte prüfen')

## Aufgabe 8.2 — Die Nachbarn der Wohnung I

*Begriffe: Rohabstand und standardisierter Abstand (Lektion 8.2, Gleichung 8.2).* Die Wohnung I (Datensatz D2.E2) hat als `floor_area` 75 m² und als `distance` 3,5 km: Das ist der Vektor `I`. Sie wird mit den sechs Referenzverkäufen und $k = 3$ geschätzt.

- `floor_area` — Fläche der Wohnung, in m² (Spalte `surface_m2` der CSV-Datei);
- `distance` — Entfernung zum Zentrum, in km (Spalte `distance_km` der CSV-Datei);
- `price` — Preise der Referenzverkäufe, in k€; $z$-Werte ohne Einheit.

### a) bis e) Die Rechnung von Hand

a) Differenzen (`floor_area`; `distance`) und Rohabstände zu I: A $(-43 ; -0{,}5)$, 43,00; B $(-30 ; -2)$, 30,07; C $(-21 ; -1)$, 21,02; D $(-5 ; -2{,}5)$, 5,59; E $(10 ; -1)$, 10,05; F $(23 ; -3)$, 23,19. Die Nachbarn sind D, E und C: $(245 + 279 + 185)/3 \approx 236{,}3$ k€.

b) $z_s = (75 - 64)/22{,}80 \approx 0{,}48$; $z_d = (3{,}5 - 1{,}833)/0{,}898 \approx 1{,}86$.

c) Standardisierte Abstände: A 1,97; B 2,59; C 1,45; D 2,79; E 1,20; F 3,49. Für E: $\sqrt{(0{,}92 - 0{,}48)^2 + (0{,}74 - 1{,}86)^2} = \sqrt{0{,}19 + 1{,}25} \approx 1{,}20$. Die Nachbarn sind E, C und A: $(279 + 185 + 118)/3 = 194$ k€.

d) D ist verschwunden. Seine Fläche liegt nahe an der von I (70 m² gegenüber 75), aber D ist 1 km vom Zentrum entfernt, I dagegen 3,5 km. In Standardabweichungen wiegen diese 2,5 km $2{,}5/0{,}898 \approx 2{,}79$, viel mehr als die 5 m² ($5/22{,}80 \approx 0{,}22$). A, das ebenfalls weit vom Zentrum entfernt liegt, ersetzt es.

e) Der nächste standardisierte Nachbar, E, liegt bereits 1,20 Standardabweichungen von I entfernt: Kein Referenzverkauf ähnelt I wirklich. Eine Schätzung über Nachbarn bleibt zwischen den bekannten Preisen; sie ist unsicher für eine Wohnung außerhalb des Bereichs, den die Daten abdecken.

In [ ]:
# Lösung: Abstände von I zu den sechs Verkäufen A bis F, zuerst roh, dann standardisiert
raw_I = np.sqrt(((X - I) ** 2).sum(axis=1))
z_I = (I - mu) / sigma
std_I = np.linalg.norm(Z - z_I, axis=1)

In [ ]:
# Prüfung von a) bis c)
if raw_I is None or z_I is None or std_I is None:
    print('noch nicht ausgefüllt. Ergänzen Sie die drei Zeilen.')
else:
    show_table(raw_I, 'raw_distance')
    (n_raw, e_raw), (n_std, e_std) = nearest(raw_I), nearest(std_I)
    print(f"a) roh: {n_raw}, Schätzung {e_raw:.1f} k€")
    print(f"b) z-Werte von I: {np.round(z_I, 2)}")
    show_table(std_I, 'std_distance')
    print(f"c) standardisiert: {n_std}, Schätzung {e_std:.0f} k€")
    print('→', 'richtig' if n_raw == ['D', 'E', 'C'] and abs(e_raw - 236.33) < 0.01
          and n_std == ['E', 'C', 'A'] and abs(e_std - 194) < 1e-9 else 'bitte prüfen')

## Aufgabe 8.3 — Meter oder Kilometer?

*Begriffe: Rohabstand und Einheiten (Lektion 8.2).* Wir nehmen wieder G und die sechs Verkäufe, aber die Entfernung zum Zentrum wird in Metern gemessen: `distance_m` $= 1\,000 \times$ `distance`.

- `distance_m` — Entfernung zum Zentrum, in m;
- `distance` — Entfernung zum Zentrum, in km; `floor_area` (m²) und `price` (k€) der Referenzverkäufe.

### a) bis d) Die Rechnung von Hand

a) Differenzen der Entfernung: 500, −1 000, 0, −1 500, 0 und −2 000 m. Rohabstände zu G: A 500,9; B 1 000,1; C 8; D 1 500,0; E 23; F 2 000,3. Nachbarn C, E und A: $(185 + 279 + 118)/3 = 194$ k€.

b) In Kilometern entschied die Fläche allein; in Metern entscheidet die Entfernung: Die drei Nachbarn sind die drei Verkäufe, die 2,5 oder 3 km entfernt liegen. Die Fläche dient nur noch dazu, zwischen C und E zu entscheiden.

c) Der Mittelwert wird 1 833 m und die Standardabweichung 898 m: Beide werden mit 1 000 multipliziert. In $(x - \mu)/\sigma$ werden Zähler und Nenner mit 1 000 multipliziert: Der z-Wert ändert sich nicht. Für G: $(2\,500 - 1\,833)/898 \approx 0{,}74$, wie in Kilometern.

d) Die z-Werte sind identisch, also auch die standardisierten Abstände: Die Nachbarn bleiben C, E und B, und die Schätzung 212 k€. Die Standardisierung macht die Nachbarn unabhängig von den Einheiten.

In [ ]:
# Lösung: die Entfernung zum Zentrum in Metern, dann die Rohabstände zu G
distance_m = 1000 * reference['distance'].to_numpy(dtype=float)
G_distance_m = 1000 * G[1]
raw_G_m = np.sqrt((reference['floor_area'].to_numpy() - G[0]) ** 2 + (distance_m - G_distance_m) ** 2)

In [ ]:
# Prüfung von a), c) und d)
if distance_m is None or G_distance_m is None or raw_G_m is None:
    print('noch nicht ausgefüllt. Ergänzen Sie die drei Zeilen.')
else:
    show_table(raw_G_m, 'raw_distance_m')
    n_m, e_m = nearest(raw_G_m)
    print(f"a) in Metern: {n_m}, Schätzung {e_m:.0f} k€")
    mu_m, sigma_m = distance_m.mean(), distance_m.std()
    z_G_m = (G_distance_m - mu_m) / sigma_m
    print(f"c) Mittelwert {mu_m:.0f} m, Standardabweichung {sigma_m:.0f} m; z-Wert von G: {z_G_m:.2f} in m, {z_G[1]:.2f} in km")
    print('→', 'richtig' if n_m == ['C', 'E', 'A'] and abs(e_m - 194) < 1e-9 and abs(z_G_m - z_G[1]) < 1e-12
          else 'bitte prüfen')

## Aufgabe 8.4 — Wie viele Nachbarn?

*Begriffe: nächste Nachbarn, Wahl von $k$ (Lektion 8.2), Validierung (Lektion 6.4).* In der Reihenfolge der standardisierten Abstände zu G sind die sechs Verkäufe und ihr `price`: C (185 k€), E (279), B (172), A (118), D (245) und F (335).

- `price` — Verkaufspreis, in k€; $k$ — Anzahl der ausgewählten Nachbarn, ohne Einheit.

### a) bis c) Die Rechnung von Hand

a) $k = 1$: 185 k€. $k = 2$: $(185 + 279)/2 = 232$ k€. $k = 3$: 212 k€. $k = 4$: $(636 + 118)/4 = 188{,}5$ k€. $k = 6$: $1\,334/6 \approx 222{,}3$ k€.

b) Das ist der mittlere Preis der sechs Verkäufe: dieselbe Schätzung für jede Wohnung. Sie ignoriert die Merkmale von G vollständig.

c) $k = 3$ liefert die kleinste Abweichung: +3 k€, gegenüber −24, +23, −20,5 und +13,3 k€. Aber $k$ anhand von G zu wählen, also des vorherzusagenden Verkaufs, heißt, die Antwort anzuschauen: Das ist das Datenleck aus Kapitel 6. Man müsste $k$ auf Validierungsverkäufen wählen, zum Beispiel durch Kreuzvalidierung (Lektion 6.4), und die Schätzung dann auf Verkäufen beurteilen, die zu keiner Entscheidung beigetragen haben.

In [ ]:
# Lösung: die Schätzung für G für jeden Wert von k
estimates = {k: nearest(std_G, k)[1] for k in (1, 2, 3, 4, 6)}

In [ ]:
# Prüfung von a) und c)
if estimates is None:
    print('noch nicht ausgefüllt. Ergänzen Sie die Zeile.')
else:
    for k, e in estimates.items():
        print(f"k = {k}: {e:.1f} k€, Abweichung vom tatsächlichen Preis (209 k€): {e - 209:+.1f} k€")
    expected = {1: 185, 2: 232, 3: 212, 4: 188.5, 6: 1334 / 6}
    print('→', 'richtig' if all(abs(estimates[k] - v) < 1e-9 for k, v in expected.items()) else 'bitte prüfen')

## Aufgabe 8.5 — Abstand oder Kosinus?

*Begriffe: Abstand (Gleichung 8.1), Kosinus (Gleichung 8.3).* Wir zählen drei Wörter in vier Nachrichten des Vereins: „Sitzung“, „Angebot“ und „Ehrenamt“ (Tabelle 8.4 im Buch).

- `message` — Kennung der Nachricht, T1 bis T4;
- `meeting_count`, `offer_count`, `volunteer_count` — wie oft die Wörter „Sitzung“, „Angebot“ und „Ehrenamt“ in der Nachricht vorkommen: die drei Koordinaten ihres Vektors.

In [ ]:
# Daten von Aufgabe 8.5 (Tabelle des Buchs)
words = pd.DataFrame({
    'message':         ['T1', 'T2', 'T3', 'T4'],
    'meeting_count':   [2, 6, 1, 0],
    'offer_count':     [0, 0, 2, 1],
    'volunteer_count': [1, 3, 2, 0],
}).set_index('message')
T = {m: words.loc[m].to_numpy(dtype=float) for m in words.index}
print(words)

### a) bis d) Die Rechnung von Hand

a) $\lVert \text{T1} \rVert = \sqrt{4 + 0 + 1} = \sqrt{5} \approx 2{,}24$; $\lVert \text{T2} \rVert = \sqrt{36 + 9} = \sqrt{45} \approx 6{,}71$; $\lVert \text{T3} \rVert = \sqrt{1 + 4 + 4} = 3$; $\lVert \text{T4} \rVert = 1$.

b) T1 − T2 $= (-4 ; 0 ; -2)$, Abstand $\sqrt{20} \approx 4{,}47$. T1 − T3 $= (1 ; -2 ; -1)$, Abstand $\sqrt{6} \approx 2{,}45$. T1 − T4 $= (2 ; -1 ; 1)$, Abstand $\sqrt{6} \approx 2{,}45$. T3 und T4 liegen gleichauf und am nächsten; T2 ist am weitesten entfernt.

c) T1 · T2 $= 15$ und $\cos = 15/(\sqrt{5} \times \sqrt{45}) = 15/15 = 1$. T1 · T3 $= 4$ und $\cos = 4/(\sqrt{5} \times 3) \approx 0{,}60$. T1 · T4 $= 0$ und $\cos = 0$. T2 ist am ähnlichsten, dann T3, dann T4.

d) T2 hat genau die Proportionen von T1: dasselbe Thema, dreimal ausführlicher. Der Abstand bestraft T2 für seine Länge. T4 hat kein Wort mit T1 gemeinsam, ist aber kurz: Seine Spitze liegt nahe an der von T1. Um nach Thema zu ordnen, eignet sich der Kosinus: Er vergleicht die Anteile der Wörter.

In [ ]:
# Lösung: der Kosinus zweier Vektoren (Gleichung 8.3)
def cosine(u, v):
    return float(np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v)))

In [ ]:
# Prüfung von a) bis c)
if cosine(T['T1'], T['T2']) is None:
    print('noch nicht ausgefüllt. Ergänzen Sie die Funktion cosine.')
else:
    results = pd.DataFrame({
        'norm': [np.linalg.norm(T[m]) for m in T],
        'distance_to_T1': [np.linalg.norm(T['T1'] - T[m]) for m in T],
        'cosine_with_T1': [cosine(T['T1'], T[m]) for m in T],
    }, index=list(T)).round(2)
    print(results)
    print('→', 'richtig' if abs(cosine(T['T1'], T['T2']) - 1) < 1e-12 and abs(cosine(T['T1'], T['T3']) - 0.596) < 5e-4
          and abs(cosine(T['T1'], T['T4'])) < 1e-12 else 'bitte prüfen')

## Aufgabe 8.6 — Herausforderung: normieren, dann vergleichen

*Begriffe: Norm und Skalarprodukt (Gleichung 8.1), Kosinus (Gleichung 8.3).* Die Fragen a) und b) sind Beweise auf Papier; das Notebook prüft c) und d).

### a) bis d) Die Rechnung von Hand

a) Mit den Rechenregeln des Skalarprodukts: $(\mathbf{u} - \mathbf{v}) \cdot (\mathbf{u} - \mathbf{v}) = \mathbf{u} \cdot \mathbf{u} - 2\, \mathbf{u} \cdot \mathbf{v} + \mathbf{v} \cdot \mathbf{v} = \lVert \mathbf{u} \rVert^2 + \lVert \mathbf{v} \rVert^2 - 2\, \mathbf{u} \cdot \mathbf{v}$.

b) Wenn beide Normen 1 sind, dann gilt $\cos(\mathbf{u}, \mathbf{v}) = \mathbf{u} \cdot \mathbf{v}$, und a) ergibt $1 + 1 - 2 \cos$.

c) Normieren ändert die Kosinuswerte nicht. T2: $\sqrt{2 - 2 \times 1} = 0$; T3: $\sqrt{2 - 2 \times 0{,}596} \approx 0{,}90$; T4: $\sqrt{2 - 0} \approx 1{,}41$. Die Rangfolge ist die des Kosinus: Je größer der Kosinus, desto kleiner der Abstand.

d) $(4 ; 3)/5 = (0{,}8 ; 0{,}6)$ und $(3 ; 4)/5 = (0{,}6 ; 0{,}8)$. Direkt beträgt die Differenz $(0{,}2 ; -0{,}2)$ und der Abstand $\sqrt{0{,}08} \approx 0{,}28$. Mit b): $\sqrt{2 - 2 \times 0{,}96} = \sqrt{0{,}08} \approx 0{,}28$.

In [ ]:
# Lösung: jede Nachricht durch ihre Norm geteilt, dann die Abstände zum normierten T1
normalized = {m: T[m] / np.linalg.norm(T[m]) for m in T}
unit_dist = {m: float(np.linalg.norm(normalized['T1'] - normalized[m])) for m in T}

In [ ]:
# Prüfung von c) und d)
if normalized is None or unit_dist is None:
    print('noch nicht ausgefüllt. Ergänzen Sie die beiden Zeilen.')
else:
    for m in ('T2', 'T3', 'T4'):
        c = float(np.dot(T['T1'], T[m]) / (np.linalg.norm(T['T1']) * np.linalg.norm(T[m])))
        print(f"{m}: normierter Abstand {unit_dist[m]:.2f}, √(2 − 2 cos) = {np.sqrt(max(2 - 2 * c, 0)):.2f}")
    u, v = embeddings['Sitzung'] / 5, embeddings['Versammlung'] / 5
    print(f"d) Sitzung und Versammlung: direkt {np.linalg.norm(u - v):.2f}, mit b) {np.sqrt(2 - 2 * np.dot(u, v)):.2f}")
    print('→', 'richtig' if abs(unit_dist['T2']) < 1e-9 and abs(unit_dist['T3'] - 0.899) < 5e-4
          and abs(unit_dist['T4'] - np.sqrt(2)) < 1e-9 else 'bitte prüfen')